<a href="https://colab.research.google.com/github/shriyyaaa/Brain--tumor--detection/blob/main/training_and_evaluation_of_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q "gymnasium>=0.29" "gymnasium-robotics>=1.3" "mujoco>=2.3.3" "stable-baselines3>=2.3" tensorboard "imageio[ffmpeg]"


In [1]:
import os
os.environ["MUJOCO_GL"] = "egl"
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import mujoco, gymnasium as gym, gymnasium_robotics
gym.register_envs(gymnasium_robotics)
print(mujoco.__version__)
env = gym.make("FetchReach-v4")
env.reset()
print("OK")

3.17.0
OK


AdroitHandRelocateDense-v1, AdroitHandHammerDense-v1, AdroitHandDoorDense-v1 environment's reward functions were updated in v1.2.1 without an environment version update. Therefore, use gymnasium-robotics==1.2.0 for v1 reproducibility or use v2 in gymnasium-robotics>=1.4.3. See https://github.com/Farama-Foundation/Gymnasium-Robotics/pull/220 for more details


In [2]:
#train

%%writefile train.py
import argparse
import os

import gymnasium as gym
import gymnasium_robotics
from stable_baselines3 import SAC, HerReplayBuffer
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback
from stable_baselines3.common.env_util import make_vec_env

gym.register_envs(gymnasium_robotics)


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--env", default="FetchPickAndPlace-v4")
    p.add_argument("--steps", type=int, default=3_000_000)
    p.add_argument("--n-envs", type=int, default=4)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--out", default="runs")
    args = p.parse_args()

    run_dir = os.path.join(args.out, f"{args.env}_seed{args.seed}")
    os.makedirs(run_dir, exist_ok=True)

    env = make_vec_env(args.env, n_envs=args.n_envs, seed=args.seed)
    eval_env = make_vec_env(args.env, n_envs=1, seed=args.seed + 100)

    model = SAC(
        "MultiInputPolicy",
        env,
        replay_buffer_class=HerReplayBuffer,
        replay_buffer_kwargs=dict(n_sampled_goal=4, goal_selection_strategy="future"),
        learning_rate=1e-3,
        buffer_size=1_000_000,
        batch_size=512,
        gamma=0.95,
        tau=0.05,
        train_freq=1,
        gradient_steps=-1,
        learning_starts=2_000,
        policy_kwargs=dict(net_arch=[512, 512, 512]),
        tensorboard_log=os.path.join(run_dir, "tb"),
        seed=args.seed,
        verbose=1,
    )

    callbacks = [
        EvalCallback(
            eval_env,
            best_model_save_path=os.path.join(run_dir, "best"),
            log_path=os.path.join(run_dir, "eval"),
            eval_freq=max(10_000 // args.n_envs, 1),
            n_eval_episodes=20,
            deterministic=True,
        ),
        CheckpointCallback(
            save_freq=max(100_000 // args.n_envs, 1),
            save_path=os.path.join(run_dir, "ckpt"),
        ),
    ]
    model.learn(total_timesteps=args.steps, callback=callbacks)
    model.save(os.path.join(run_dir, "final"))


if __name__ == "__main__":
    main()

Overwriting train.py


In [5]:
#evaluate

%%writefile evaluate.py
import argparse

import gymnasium as gym
import gymnasium_robotics
import imageio
import numpy as np
from stable_baselines3 import SAC

gym.register_envs(gymnasium_robotics)


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--model", required=True)
    p.add_argument("--env", default="FetchPickAndPlace-v4")
    p.add_argument("--episodes", type=int, default=100)
    p.add_argument("--video", default=None)
    args = p.parse_args()

    env = gym.make(args.env, render_mode="rgb_array" if args.video else None)
    model = SAC.load(args.model)

    frames, successes = [], []
    for ep in range(args.episodes):
        obs, _ = env.reset()
        done, info = False, {}
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, _, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            if args.video and ep < 5:
                frames.append(env.render())
        successes.append(float(info.get("is_success", 0.0)))

    print(f"Success rate over {args.episodes} episodes: {np.mean(successes):.1%}")
    if args.video and frames:
        imageio.mimsave(args.video, frames, fps=30)
        print(f"Saved {args.video}")


if __name__ == "__main__":
    main()

Overwriting evaluate.py


In [6]:
!sed -i 's/mujoco\.mjtJoint\.\(mjJNT_[A-Z]*\)/int(mujoco.mjtJoint.\1)/g' /usr/local/lib/python3.13/dist-packages/gymnasium_robotics/utils/mujoco_utils.py
!grep -n "mjtJoint" /usr/local/lib/python3.13/dist-packages/gymnasium_robotics/utils/mujoco_utils.py


137:    if joint_type == int(int(int(mujoco.mjtJoint.mjJNT_FREE))):
139:    elif joint_type == int(int(int(mujoco.mjtJoint.mjJNT_BALL))):
142:        assert joint_type in (int(int(int(mujoco.mjtJoint.mjJNT_HINGE))), int(int(int(mujoco.mjtJoint.mjJNT_SLIDE))))
162:    if joint_type == int(int(int(mujoco.mjtJoint.mjJNT_FREE))):
164:    elif joint_type == int(int(int(mujoco.mjtJoint.mjJNT_BALL))):
167:        assert joint_type in (int(int(int(mujoco.mjtJoint.mjJNT_HINGE))), int(int(int(mujoco.mjtJoint.mjJNT_SLIDE))))
187:    if joint_type == int(int(int(mujoco.mjtJoint.mjJNT_FREE))):
189:    elif joint_type == int(int(int(mujoco.mjtJoint.mjJNT_BALL))):
192:        assert joint_type in (int(int(int(mujoco.mjtJoint.mjJNT_HINGE))), int(int(int(mujoco.mjtJoint.mjJNT_SLIDE))))
208:    if joint_type == int(int(int(mujoco.mjtJoint.mjJNT_FREE))):
210:    elif joint_type == int(int(int(mujoco.mjtJoint.mjJNT_BALL))):
213:        assert joint_type in (int(int(int(mujoco.mjtJoint.mjJNT_HINGE))), int(

In [14]:
!pip uninstall -y mujoco
!pip install -q "mujoco>=3.2,<4"

Found existing installation: mujoco 3.2.7
Uninstalling mujoco-3.2.7:
  Successfully uninstalled mujoco-3.2.7


In [ ]:
!python train.py --env FetchReach-v4 --steps 50000 --out /content/drive/MyDrive/rl_runs

AdroitHandRelocateDense-v1, AdroitHandHammerDense-v1, AdroitHandDoorDense-v1 environment's reward functions were updated in v1.2.1 without an environment version update. Therefore, use gymnasium-robotics==1.2.0 for v1 reproducibility or use v2 in gymnasium-robotics>=1.4.3. See https://github.com/Farama-Foundation/Gymnasium-Robotics/pull/220 for more details
I0000 00:00:1795533414.731461    3592 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1795533414.770483    3592 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795533416.601099    3592 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functi

In [ ]:
!python evaluate.py --model /content/drive/MyDrive/rl_runs/FetchReach-v4_seed0/best/best_model.zip --env FetchReach-v4 --episodes 20 --video demo.mp4

from IPython.display import Video
Video("demo.mp4", embed=True)